# 28 — Segmentación Débilmente Supervisada (WSSS) y Alineación Anatómica de Máscaras ROI


---

## Objetivo

En este notebook se formaliza y evalúa la Segmentación Semántica Débilmente Supervisada (WSSS - *Weakly Supervised Semantic Segmentation*) sobre mamografías clínicas complejas, utilizando exclusivamente las señales de activación continua inducidas por clasificadores profundos entrenados a nivel de imagen ($y \in \{0, 1\}$).

### 1. Corrección en la alineación de Máscaras Ground-Truth (CBIS-DDSM)
En la base de datos pública CBIS-DDSM, el catálogo DICOM (`dicom_info.csv`) asocia a cada lesión dos registros bajo el mismo identificador de serie (`SeriesInstanceUID`):
1. Un recorte local magnificado de la lesión (*cropped image*, e.g. $500 \times 500$ px).
2. La máscara binaria radiológica de resolución completa (*ROI mask image*, e.g. $3000 \times 5000$ px) correspondiente a la mamografía completa (*full mammogram*).

> Problema: Un cargador ingenuo que indexe por serie sin discriminar `SeriesDescription == 'ROI mask images'` suele sobreescribir la máscara completa con el recorte magnificado. Al redimensionar a $224 \times 224$, el recorte se estira ocupando la totalidad del encuadre, distorsionando el centroide real y la geometría clínica.  
> Solución (`roi_loader.py`): Filtrado explícito por tipo de serie y verificación estricta de igualdad de dimensiones $(\text{ancho}_{\text{img}} == \text{ancho}_{\text{roi}}, \text{alto}_{\text{img}} == \text{alto}_{\text{roi}})$, garantizando una correspondencia espacial 1:1.

### 2. Máscara de mama $\Omega_{\mathrm{breast}}$
Para evitar que activaciones residuales o gradientes en el fondo negro de aire se traduzcan en falsos positivos (como se observa característicamente en casos con artefactos periféricos, e.g. `P_00562`), se define la máscara del parénquima mamario mediante binarización de Otsu sobre intensidades radiográficas y clausura morfológica con relleno de huecos en retículos booleanos:
$$\Omega_{\mathrm{breast}} = \phi_B\big(\gamma_B(\mathcal{B}_\tau(I))\big) \subset \Omega$$
El mapa continuo de atribución enmascarado se define entonces como $H_{\mathrm{tisular}}(u, v) = H(u, v) \cdot \mathbb{I}_{\Omega_{\mathrm{breast}}}(u, v)$, anulando de forma determinista cualquier activación espuria fuera de la mama.

### 3. Núcleo focal ($\ge 60\%$, Tonalidad Amarilla-Roja)
En lugar de binarizaciones permisivas que capturan grandes extensiones de parénquima sano en tonalidad verde ($\ge 20\%$), el operador de Núcleo Focal fija el umbral en el percentil 85 de los vóxeles activos en tejido o al menos el $60\%$ de la intensidad pico:
$$\tau_{\mathrm{focal}} = \max\big(0.60 \cdot \max H_{\mathrm{tisular}}, \; \mathrm{Percentil}_{85}(H_{\mathrm{tisular}} \mid \Omega_{\mathrm{breast}})\big)$$
seguido de apertura ($	ext{struct}=3\times 3$) y cierre morfológico centrado en la componente conexa del punto crítico $\arg\max H_{\mathrm{tisular}}$.


In [ ]:
# 1. Configuración del entorno e importaciones
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from IPython.display import Image as IPImage, display

CODIGOS = os.environ.get("TESIS_CODIGOS") or os.path.abspath(os.path.join(os.getcwd(), ".."))   # raiz del repo (ejecutar desde pipeline/)
sys.path.insert(0, CODIGOS)

from src.data.roi_loader import get_roi_manifest, CBISROIDataset
from src.explainability.weak_segmentation import (
    extract_breast_tissue_mask,
    segment_wsss_focal_core,
    segment_otsu,
    segment_fixed_threshold,
    compute_centroid,
    compute_segmentation_metrics
)
from src.reporting.clinical_evidence_extractor import determine_quadrant

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REPORTS_DIR = os.path.join(CODIGOS, "reports", "experiments")
FIGS_DIR = os.path.join(CODIGOS, "reports", "figures")

print(f"Entorno configurado correctamente. Dispositivo: {DEVICE}")

## 1. Verificación de la Carga de Máscaras Clínicas Ground-Truth 1:1

Verificamos que el dataset filtrado de prueba de CBIS-DDSM cuenta con correspondencia exacta de resolución entre la imagen original y la máscara patológica anotada por radiólogos.


In [ ]:
# Cargar el manifiesto con la correspondencia verificada
df_roi = get_roi_manifest(CODIGOS)
ds = CBISROIDataset(df_roi, image_size=224)

print(f"Total mamografías clínicas de prueba con ROI ground-truth: {len(ds)}")
print("Distribución por patología:")
print(df_roi["pathology"].value_counts())
print("\nDistribución por tipo de anomalía:")
print(df_roi["abnormality_type"].value_counts())

# Inspeccionar una muestra aleatoria de casos
sample_indices = [0, 2, 10, 27]
for s_idx in sample_indices:
    item = ds[s_idx]
    gt = item["mask_np"]
    cy, cx = compute_centroid(gt)
    print(f"Caso {s_idx} ({item['patient_id']} - {item['pathology']}): Área GT={int(gt.sum())} px ({(gt.sum()/(224*224))*100:.2f}%), Centroide=({cx:.1f}, {cy:.1f})")

## 2. Demostración Interactiva: Extracción de Tejido Mamario y Segmentación Focal

Examinamos el pipeline propuesto paso a paso en cuatro casos característicos:
1. `P_00202`: Masa maligna espiculada.
2. `P_00562`: Caso crítico donde un clasificador estándar activa sobre el fondo de aire negro, pero la máscara tisular suprime totalmente dicho falso positivo.
3. `P_01502`: Masa benigna circunscrita.
4. `P_00141`: Calcificaciones benignas (con discordancia histórica BI-RADS especialista 4 vs predicción modelo 2).


In [ ]:
# Cargar modelo ResNet-18 Robusto
rob_model = models.resnet18(weights=None)
rob_model.fc = nn.Linear(512, 2)
ckpt_rob = torch.load(os.path.join(CODIGOS, "reports", "models", "resnet18_robust_best.pt"), map_location=DEVICE)
rob_model.load_state_dict(ckpt_rob["model_state_dict"])
rob_model = rob_model.to(DEVICE).eval()

class GradCAM:
    def __init__(self, model, target_layer, device):
        self.model = model
        self.target_layer = target_layer
        self.device = device
        self.activations = None
        self.gradients = None
        self.target_layer.register_forward_hook(self._fwd_hook)
        self.target_layer.register_full_backward_hook(self._bwd_hook)
    def _fwd_hook(self, m, i, o):
        self.activations = o
    def _bwd_hook(self, m, gi, go):
        self.gradients = go[0]
    def __call__(self, x, class_idx=1):
        self.model.zero_grad()
        out = self.model(x)
        out[0, class_idx].backward(retain_graph=False)
        weights = self.gradients[0].mean(dim=(1, 2))
        cam = (self.activations[0] * weights.view(-1, 1, 1)).sum(dim=0)
        cam = F.relu(cam).detach().cpu().numpy()
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        else:
            cam = np.zeros_like(cam)
        cam_img = Image.fromarray((cam * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)
        return np.array(cam_img, dtype=np.float32) / 255.0

cam_extractor = GradCAM(rob_model, rob_model.layer4, DEVICE)

# Casos de estudio interactivo
demo_cases = [2, 10, 27, 0]  # P_00202, P_00562, P_01502, P_00141
fig, axes = plt.subplots(len(demo_cases), 4, figsize=(18, 4.2 * len(demo_cases)))
plt.subplots_adjust(wspace=0.08, hspace=0.25)

col_titles = [
    "Mamografía + ROI Ground Truth",
    "Máscara Tisular $\Omega_{\mathrm{breast}}$",
    "Grad-CAM Enmascarado en Tejido",
    "Núcleo Focal (≥60%) + Cuadrante"
]
for c, t in enumerate(col_titles):
    axes[0, c].set_title(t, fontsize=12, fontweight='bold', pad=10)

for r, idx in enumerate(demo_cases):
    item = ds[idx]
    img_t = item["image_tensor"].unsqueeze(0).to(DEVICE)
    gt_mask = item["mask_np"]
    pid = item["patient_id"]
    patho = item["pathology"]
    
    img_raw = item["image_tensor"].cpu().numpy()[0]
    img_disp = (img_raw - img_raw.min()) / (img_raw.max() - img_raw.min() + 1e-8)
    
    # Pipeline WSSS
    tissue_mask = extract_breast_tissue_mask(img_disp)
    raw_cam = cam_extractor(img_t, class_idx=1)
    focal_mask, cam_tissue, peak = segment_wsss_focal_core(raw_cam, tissue_mask, min_tau_ratio=0.60)
    
    cy_gt, cx_gt = compute_centroid(gt_mask)
    cy_pred, cx_pred = compute_centroid(focal_mask)
    quad = determine_quadrant(cy_pred, cx_pred, "LEFT", "MLO")
    m = compute_segmentation_metrics(focal_mask, gt_mask)
    
    # Col 1: Original + GT
    ax = axes[r, 0]
    ax.imshow(img_disp, cmap="gray")
    if gt_mask.sum() > 0:
        ax.contour(gt_mask, levels=[0.5], colors=['#F59E0B'], linewidths=2.2)
    ax.plot(cx_gt, cy_gt, "o", color="#F59E0B", markersize=6)
    ax.set_ylabel(f"{pid}\n({patho})", fontsize=11, fontweight='bold')
    ax.axis("off")
    
    # Col 2: Máscara tisular
    ax = axes[r, 1]
    ax.imshow(img_disp, cmap="gray")
    ax.contour(tissue_mask, levels=[0.5], colors=['#06B6D4'], linewidths=1.8)
    ax.text(0.04, 0.92, "Parénquima Activo", transform=ax.transAxes, fontsize=8.5, color="#FFFFFF",
            bbox=dict(boxstyle="round,pad=0.25", fc="#0891B2", alpha=0.9))
    ax.axis("off")
    
    # Col 3: CAM enmascarado
    ax = axes[r, 2]
    ax.imshow(img_disp, cmap="gray")
    cam_show = raw_cam * tissue_mask
    alpha_map = np.clip((cam_show - 0.05) / 0.95, 0.0, 0.65) * (tissue_mask > 0.5)
    cmap = plt.cm.jet
    cam_rgba = cmap(cam_show)
    cam_rgba[..., 3] = alpha_map
    ax.imshow(cam_rgba)
    ax.plot(peak[1], peak[0], "w*", markersize=10, markeredgecolor="#EA580C")
    ax.axis("off")
    
    # Col 4: Núcleo focal
    ax = axes[r, 3]
    ax.imshow(img_disp, cmap="gray")
    if focal_mask.sum() > 0:
        ax.contour(focal_mask, levels=[0.5], colors=['#EA580C'], linewidths=2.2)
    ax.plot(cx_pred, cy_pred, "+", color="#F97316", markersize=14, markeredgewidth=2.2)
    ax.text(0.04, 0.92, f"{quad} (Dist={m['centroid_dist']:.1f}px)", transform=ax.transAxes, fontsize=8.5, color="#FFFFFF",
            bbox=dict(boxstyle="round,pad=0.25", fc="#9A3412", alpha=0.92))
    ax.axis("off")

plt.tight_layout()
plt.show()

## 3. Evaluación Cuantitativa del Benchmark en la Cohorte Completa (368 Casos)

Cargamos los resultados consolidados tras evaluar los tres paradigmas de extracción convolucional cruzados con las estrategias de binarización y el nuevo operador de Núcleo Focal con Máscara Tisular.


In [ ]:
summary_file = os.path.join(REPORTS_DIR, "28_weak_segmentation_summary.json")
with open(summary_file, "r", encoding="utf-8") as f:
    summary_data = json.load(f)

df_summary = pd.DataFrame(summary_data)
cols_rename = {
    "Paradigma": "Paradigma de Atribución",
    "Método": "Estrategia de Segmentación",
    "Dist_Centroide_Mal": "Dist. Centroide Malignos (px)",
    "Precision_Cuadrante_Mal": "Precisión Cuadrante Malignos (%)",
    "Dist_Centroide": "Dist. Centroide Global (px)",
    "Precision_Cuadrante": "Precisión Cuadrante Global (%)",
    "Recall_Global": "Recall / Cobertura Tumoral (%)"
}

df_table = df_summary.rename(columns=cols_rename).sort_values("Dist. Centroide Malignos (px)", ascending=True)
display(df_table[[
    "Paradigma de Atribución",
    "Estrategia de Segmentación",
    "Dist. Centroide Malignos (px)",
    "Precisión Cuadrante Malignos (%)",
    "Dist. Centroide Global (px)",
    "Precisión Cuadrante Global (%)"
]].style.format({
    "Dist. Centroide Malignos (px)": "{:.1f}",
    "Precisión Cuadrante Malignos (%)": "{:.1f} %",
    "Dist. Centroide Global (px)": "{:.1f}",
    "Precisión Cuadrante Global (%)": "{:.1f} %"
}).background_gradient(subset=["Dist. Centroide Malignos (px)"], cmap="viridis_r"))

### Gráficos comparativos


In [ ]:
bars_img = os.path.join(FIGS_DIR, "28_weakly_supervised_metrics_bars.png")
if os.path.exists(bars_img):
    display(IPImage(filename=bars_img, width=950))
else:
    print("Figura de barras no encontrada.")

## 4. Galería Visual Multi-Caso y Diagnóstico Topológico de Error


In [ ]:
gallery_img = os.path.join(FIGS_DIR, "28_weakly_supervised_masks_gallery.png")
if os.path.exists(gallery_img):
    display(IPImage(filename=gallery_img, width=950))
else:
    print("Figura de galería no encontrada.")

## 5. Discusión

### Nota sobre Dice en mamografia
Al calcular métricas conjuntistas sobre mamografías completas con resolución normalizada ($224 \times 224$ px), las lesiones patológicas (especialmente microcalcificaciones agrupadas) abarcan típicamente entre el $0.2\%$ y el $2.5\%$ del área total de la imagen (menos de $100$ píxeles). En contraste, el campo receptivo efectivo de la última capa convolucional (`layer4`) es intrínsecamente suave y difuso debido al escalado bilineal.

Por consiguiente:
- El Coeficiente Dice estricto a nivel de píxel penaliza drásticamente cualquier halo peritumoral, oscilando en valores bajos ($0.02 - 0.05$) frente a lesiones milimétricas.
- Las métricas clínicamente trascendentes son la Distancia Euclidiana al Centroide Tumoral ($d(c_{M^*}, c_{\hat{M}})$) y la Concordancia de Cuadrante Anatómico, pues en el protocolo radiológico asistencial el informe médico exige ubicar la lesión en su cuadrante y alertar sobre la sospecha morfológica.

### Núcleo focal con mascara tisular
1. Mínima distancia de centroide en lesiones malignas: Logra reducir la distancia centroide-lesión a $64.4$ píxeles (frente a $> 82$ píxeles del baseline y de Otsu sin filtro de tejido).
2. Máxima concordancia de cuadrante: Alcanza un $38.7\%$ de exactitud de cuadrante en lesiones malignas (superando en más de $8$ puntos porcentuales a los métodos sin enmascaramiento tisular).
3. Eliminación total de artefactos de fondo de aire: Como demuestra el caso `P_00562`, ningún gradiente residual en el fondo negro supera el filtro $\Omega_{\mathrm{breast}}$, protegiendo al radiólogo de falsas alarmas en regiones extracorpóreas.

Este pipeline alimenta de manera determinista los tokens espaciales y descriptores geométricos empleados en el reporte radiológico de la estación de trabajo Prometeo PACS.
